# Statistical Arbitrage Analysis

Intraday mean-reversion pairs trading system for US equities.

**Workflow:**
1. Data must be fetched first: `python scripts/fetch_data.py`
2. Preprocess data (resample to multiple timeframes)
3. Discover cointegrated pairs on daily data
4. Generate signals on intraday data
5. Backtest and analyze performance

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import numpy as np

from utils.config import CONFIG, get_all_tickers, get_all_sectors
from analysis.preprocessing import (
    preprocess_all_tickers,
    load_processed,
    load_raw,
)
from analysis.cointegration import (
    find_cointegrated_pairs,
    save_cointegration_results,
)
from analysis.spread import build_spread_frame
from strategy.signals import generate_signals, apply_signals_with_holding
from strategy.backtest import backtest_pair, backtest_all_pairs, save_trades
from strategy.portfolio import (
    build_equity_curve,
    build_daily_returns,
    monthly_returns_table,
    pair_performance_summary,
)
from utils.metrics import calculate_all_metrics

print(f"Polars version: {pl.__version__}")
print(f"Project root:   {project_root}")
print(f"Data directory:  {CONFIG['data_dir']}")

## 2. Data Loading & Preprocessing

In [ ]:
# Preprocess all raw 1-min data into multiple timeframes
# This reads from data/raw/1min/ and writes to data/processed/{timeframe}/
# Only need to run once (or when raw data changes)

summary = preprocess_all_tickers()
print(summary)

In [ ]:
# Show available tickers and data summary
raw_dir = Path(CONFIG["raw_dir"])
available_tickers = sorted(p.stem for p in raw_dir.glob("*.parquet"))
print(f"Available tickers: {len(available_tickers)}")
print(available_tickers)

# Quick data quality check on first ticker
if available_tickers:
    sample_ticker = available_tickers[0]
    df_sample = load_processed(sample_ticker, "daily")
    print(f"\n--- {sample_ticker} daily data ---")
    print(f"Shape: {df_sample.shape}")
    print(f"Date range: {df_sample['timestamp'].min()} to {df_sample['timestamp'].max()}")
    print(df_sample.describe())

In [ ]:
# Verify data quality: check for missing bars and outliers
quality_report = []
for ticker in available_tickers:
    try:
        df = load_processed(ticker, "1min")
        daily = load_processed(ticker, "daily")
        quality_report.append({
            "ticker": ticker,
            "1min_bars": len(df),
            "daily_bars": len(daily),
            "min_close": df["close"].min(),
            "max_close": df["close"].max(),
            "null_count": df.null_count().sum_horizontal()[0],
        })
    except Exception as e:
        print(f"Error loading {ticker}: {e}")

quality_df = pl.DataFrame(quality_report)
print(quality_df)

## 3. Pair Discovery (Cointegration Analysis)

In [ ]:
# Run cointegration analysis on daily data
coint_pairs = find_cointegrated_pairs()
print(f"Found {len(coint_pairs)} cointegrated pairs")

# Save results
if len(coint_pairs) > 0:
    save_cointegration_results(coint_pairs)

In [ ]:
# Display top 20 pairs by cointegration strength
if len(coint_pairs) > 0:
    top_20 = coint_pairs.head(20)
    print("Top 20 Cointegrated Pairs (by p-value):")
    print(top_20)
else:
    print("No cointegrated pairs found. Check data and thresholds.")

In [ ]:
# Visualize top pairs: scatter plots of prices
if len(coint_pairs) >= 3:
    n_plots = min(6, len(coint_pairs))
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[f"{r['ticker_a']}/{r['ticker_b']} (p={r['p_value']:.4f})"
                        for r in coint_pairs.head(n_plots).iter_rows(named=True)]
    )
    for idx, row in enumerate(coint_pairs.head(n_plots).iter_rows(named=True)):
        r, c = divmod(idx, 3)
        df_a = load_processed(row["ticker_a"], "daily")
        df_b = load_processed(row["ticker_b"], "daily")
        merged = df_a.select(["timestamp", "close"]).rename({"close": "A"}).join(
            df_b.select(["timestamp", "close"]).rename({"close": "B"}),
            on="timestamp", how="inner"
        )
        fig.add_trace(
            go.Scatter(x=merged["A"].to_list(), y=merged["B"].to_list(),
                       mode="markers", marker=dict(size=3), showlegend=False),
            row=r+1, col=c+1
        )
    fig.update_layout(height=600, title_text="Price Scatter Plots - Top Cointegrated Pairs")
    fig.show()

In [ ]:
# Spread plots for top pairs (daily)
if len(coint_pairs) >= 1:
    n_plots = min(4, len(coint_pairs))
    fig = make_subplots(rows=n_plots, cols=1,
                        subplot_titles=[f"{r['ticker_a']}/{r['ticker_b']} spread"
                                        for r in coint_pairs.head(n_plots).iter_rows(named=True)])
    for idx, row in enumerate(coint_pairs.head(n_plots).iter_rows(named=True)):
        df_a = load_processed(row["ticker_a"], "daily")
        df_b = load_processed(row["ticker_b"], "daily")
        spread_df = build_spread_frame(df_a, df_b, row["hedge_ratio"], window=20)
        fig.add_trace(
            go.Scatter(x=spread_df["timestamp"].to_list(), y=spread_df["spread"].to_list(),
                       name="Spread", line=dict(width=1)),
            row=idx+1, col=1
        )
        fig.add_trace(
            go.Scatter(x=spread_df["timestamp"].to_list(), y=spread_df["rolling_mean"].to_list(),
                       name="Mean", line=dict(dash="dash", width=1)),
            row=idx+1, col=1
        )
    fig.update_layout(height=250*n_plots, title_text="Spread Time Series - Top Pairs")
    fig.show()

## 4. Signal Analysis

In [ ]:
# Generate signals on 1-min data for top 5 pairs
if len(coint_pairs) >= 1:
    top_5 = coint_pairs.head(5)
    for row in top_5.iter_rows(named=True):
        ticker_a, ticker_b = row["ticker_a"], row["ticker_b"]
        hedge_ratio = row["hedge_ratio"]
        print(f"\n=== {ticker_a}/{ticker_b} (hedge={hedge_ratio:.4f}) ===")

        df_a = load_processed(ticker_a, "1min")
        df_b = load_processed(ticker_b, "1min")
        spread_df = build_spread_frame(df_a, df_b, hedge_ratio)
        spread_df = spread_df.drop_nulls(subset=["z_score"])

        signals = generate_signals(spread_df)
        n_long = signals.filter(pl.col("entry_long")).height
        n_short = signals.filter(pl.col("entry_short")).height
        print(f"  Raw entry signals: {n_long} long, {n_short} short")
        print(f"  Z-score range: [{spread_df['z_score'].min():.2f}, {spread_df['z_score'].max():.2f}]")
        print(f"  Z-score mean: {spread_df['z_score'].mean():.4f}, std: {spread_df['z_score'].std():.4f}")

In [ ]:
# Plot spread with z-score bands and entry/exit points for top pair
if len(coint_pairs) >= 1:
    row = coint_pairs.row(0, named=True)
    ticker_a, ticker_b = row["ticker_a"], row["ticker_b"]

    df_a = load_processed(ticker_a, "1min")
    df_b = load_processed(ticker_b, "1min")
    spread_df = build_spread_frame(df_a, df_b, row["hedge_ratio"])
    spread_df = spread_df.drop_nulls(subset=["z_score"])

    signals = generate_signals(spread_df)
    positioned = apply_signals_with_holding(signals)

    ts = spread_df["timestamp"].to_list()
    zs = spread_df["z_score"].to_list()

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts, y=zs, name="Z-Score", line=dict(width=0.8)))
    fig.add_hline(y=CONFIG["z_score_entry"], line_dash="dash", line_color="red",
                  annotation_text="Entry Short")
    fig.add_hline(y=-CONFIG["z_score_entry"], line_dash="dash", line_color="green",
                  annotation_text="Entry Long")
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.add_hline(y=CONFIG["z_score_stop"], line_dash="dot", line_color="orange",
                  annotation_text="Stop")
    fig.add_hline(y=-CONFIG["z_score_stop"], line_dash="dot", line_color="orange")

    # Mark entries and exits
    entries = positioned.filter(pl.col("entry_bar"))
    exits = positioned.filter(pl.col("exit_bar"))
    fig.add_trace(go.Scatter(
        x=entries["timestamp"].to_list(), y=entries["z_score"].to_list(),
        mode="markers", marker=dict(symbol="triangle-up", size=8, color="green"),
        name="Entry"
    ))
    fig.add_trace(go.Scatter(
        x=exits["timestamp"].to_list(), y=exits["z_score"].to_list(),
        mode="markers", marker=dict(symbol="x", size=8, color="red"),
        name="Exit"
    ))

    fig.update_layout(
        title=f"Z-Score with Signals: {ticker_a}/{ticker_b}",
        xaxis_title="Time", yaxis_title="Z-Score",
        height=500,
    )
    fig.show()

In [ ]:
# Distribution of z-scores for top pair
if len(coint_pairs) >= 1:
    row = coint_pairs.row(0, named=True)
    df_a = load_processed(row["ticker_a"], "1min")
    df_b = load_processed(row["ticker_b"], "1min")
    spread_df = build_spread_frame(df_a, df_b, row["hedge_ratio"])
    spread_df = spread_df.drop_nulls(subset=["z_score"])

    fig = px.histogram(
        x=spread_df["z_score"].to_list(),
        nbins=100,
        title=f"Z-Score Distribution: {row['ticker_a']}/{row['ticker_b']}",
        labels={"x": "Z-Score", "y": "Count"},
    )
    fig.add_vline(x=CONFIG["z_score_entry"], line_dash="dash", line_color="red")
    fig.add_vline(x=-CONFIG["z_score_entry"], line_dash="dash", line_color="green")
    fig.show()

## 5. Backtesting

In [ ]:
# Run backtest on top 10 pairs using 1-min data
if len(coint_pairs) >= 1:
    all_trades = backtest_all_pairs(coint_pairs, timeframe="1min")
    print(f"Total trades: {len(all_trades)}")

    if len(all_trades) > 0:
        save_trades(all_trades)
        print("\nFirst 20 trades:")
        print(all_trades.head(20))
else:
    print("No pairs to backtest.")

In [ ]:
# Trade statistics by pair
if len(all_trades) > 0:
    pair_summary = pair_performance_summary(all_trades)
    print("Per-Pair Performance:")
    print(pair_summary)

    # Overall trade stats
    print(f"\n--- Overall Trade Statistics ---")
    print(f"Total trades:       {len(all_trades)}")
    print(f"Avg holding (min):  {all_trades['holding_minutes'].mean():.1f}")
    print(f"Avg PnL (net):      {all_trades['pnl_net'].mean():.4f}")
    print(f"Total PnL (net):    {all_trades['pnl_net'].sum():.2f}")

    # Exit reason breakdown
    print("\nExit reasons:")
    print(all_trades.group_by("reason_exit").len().sort("len", descending=True))

## 6. Performance Analysis

In [ ]:
# Build equity curve and calculate metrics
if len(all_trades) > 0:
    equity = build_equity_curve(all_trades)
    metrics = calculate_all_metrics(all_trades, equity["equity"])

    print("=== Performance Metrics ===")
    for k, v in metrics.items():
        if k != "exit_reason_breakdown":
            print(f"  {k:25s}: {v}")
    if "exit_reason_breakdown" in metrics:
        print(f"  {'exit_reason_breakdown':25s}: {metrics['exit_reason_breakdown']}")

In [ ]:
# Plot cumulative P&L curve
if len(all_trades) > 0 and len(equity) > 0:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=equity["timestamp"].to_list(),
        y=equity["equity"].to_list(),
        name="Equity",
        line=dict(width=1.5),
    ))
    fig.add_hline(y=CONFIG["capital"], line_dash="dot", line_color="gray",
                  annotation_text="Starting Capital")
    fig.update_layout(
        title="Portfolio Equity Curve",
        xaxis_title="Time",
        yaxis_title="Equity ($)",
        height=400,
    )
    fig.show()

In [ ]:
# Plot drawdown curve
if len(all_trades) > 0 and len(equity) > 0:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=equity["timestamp"].to_list(),
        y=(equity["drawdown"] * 100).to_list(),
        fill="tozeroy",
        name="Drawdown",
        line=dict(width=1, color="red"),
    ))
    fig.update_layout(
        title="Drawdown (%)",
        xaxis_title="Time",
        yaxis_title="Drawdown (%)",
        height=300,
    )
    fig.show()

In [ ]:
# Monthly returns heatmap
if len(all_trades) > 0 and len(equity) > 0:
    monthly = monthly_returns_table(equity)
    if len(monthly) > 0:
        pivot = monthly.pivot(
            on="month", index="year", values="monthly_return_pct"
        ).sort("year")

        month_names = ["Jan","Feb","Mar","Apr","May","Jun",
                       "Jul","Aug","Sep","Oct","Nov","Dec"]
        cols = [c for c in pivot.columns if c != "year"]
        z_data = pivot.select(cols).to_numpy()
        years = pivot["year"].to_list()

        fig = go.Figure(data=go.Heatmap(
            z=z_data,
            x=[month_names[int(c)-1] if c.isdigit() else c for c in cols],
            y=[str(y) for y in years],
            colorscale="RdYlGn",
            text=np.round(z_data, 2),
            texttemplate="%{text}%",
            hovertemplate="Month: %{x}<br>Year: %{y}<br>Return: %{z:.2f}%<extra></extra>",
        ))
        fig.update_layout(title="Monthly Returns Heatmap (%)", height=300)
        fig.show()

In [ ]:
# Distribution of trade returns
if len(all_trades) > 0:
    fig = px.histogram(
        x=all_trades["pnl_net"].to_list(),
        nbins=50,
        title="Distribution of Trade Returns (Net P&L)",
        labels={"x": "P&L ($)", "y": "Count"},
    )
    fig.add_vline(x=0, line_dash="dash", line_color="red")
    fig.show()

## 7. Detailed Pair Analysis

In [ ]:
# Deep dive on best and worst performing pairs
if len(all_trades) > 0:
    pair_stats = pair_performance_summary(all_trades)

    if len(pair_stats) >= 1:
        best_pair = pair_stats.row(0, named=True)["pair"]
        worst_pair = pair_stats.row(-1, named=True)["pair"]

        print(f"Best pair:  {best_pair}")
        print(f"Worst pair: {worst_pair}")

        for label, pair_name in [("BEST", best_pair), ("WORST", worst_pair)]:
            pair_trades = all_trades.filter(pl.col("pair") == pair_name)
            print(f"\n--- {label}: {pair_name} ---")
            print(f"  Trades:     {len(pair_trades)}")
            print(f"  Win rate:   {(pair_trades['pnl_net'] > 0).mean():.2%}")
            print(f"  Total PnL:  {pair_trades['pnl_net'].sum():.2f}")
            print(f"  Avg PnL:    {pair_trades['pnl_net'].mean():.4f}")
            print(f"  Avg hold:   {pair_trades['holding_minutes'].mean():.1f} min")

In [ ]:
# Detailed trade chart for best pair
if len(all_trades) > 0 and len(pair_stats) >= 1:
    best_pair = pair_stats.row(0, named=True)["pair"]
    tickers = best_pair.split("/")
    if len(tickers) == 2:
        ta, tb = tickers
        bp_row = coint_pairs.filter(
            (pl.col("ticker_a") == ta) & (pl.col("ticker_b") == tb)
        )
        if len(bp_row) > 0:
            hr = bp_row.row(0, named=True)["hedge_ratio"]
            df_a = load_processed(ta, "1min")
            df_b = load_processed(tb, "1min")
            spread_df = build_spread_frame(df_a, df_b, hr)
            spread_df = spread_df.drop_nulls(subset=["z_score"])

            pair_trades = all_trades.filter(pl.col("pair") == best_pair)

            fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                                subplot_titles=["Spread", "Z-Score"],
                                vertical_spacing=0.08)

            fig.add_trace(go.Scatter(
                x=spread_df["timestamp"].to_list(), y=spread_df["spread"].to_list(),
                name="Spread", line=dict(width=0.5)),
                row=1, col=1)

            fig.add_trace(go.Scatter(
                x=spread_df["timestamp"].to_list(), y=spread_df["z_score"].to_list(),
                name="Z-Score", line=dict(width=0.5)),
                row=2, col=1)

            # Mark entries
            fig.add_trace(go.Scatter(
                x=pair_trades["entry_time"].to_list(),
                y=pair_trades["entry_spread"].to_list(),
                mode="markers", marker=dict(symbol="triangle-up", size=8, color="green"),
                name="Entry"), row=1, col=1)

            # Mark exits
            fig.add_trace(go.Scatter(
                x=pair_trades["exit_time"].to_list(),
                y=pair_trades["exit_spread"].to_list(),
                mode="markers", marker=dict(symbol="x", size=8, color="red"),
                name="Exit"), row=1, col=1)

            fig.update_layout(height=700,
                              title=f"Detailed Analysis: {best_pair}")
            fig.show()

## 8. Sensitivity Analysis

In [ ]:
# Test different z-score thresholds
if len(coint_pairs) >= 1:
    z_thresholds = [2.0, 2.5, 3.0]
    sensitivity_results = []

    for z_thresh in z_thresholds:
        print(f"\nTesting z_entry = {z_thresh}...")
        trades = backtest_all_pairs(
            coint_pairs, timeframe="1min",
            z_entry=z_thresh,
        )
        if len(trades) > 0:
            eq = build_equity_curve(trades)
            m = calculate_all_metrics(trades, eq["equity"])
            m["z_entry"] = z_thresh
            sensitivity_results.append(m)

    if sensitivity_results:
        sens_df = pl.DataFrame([
            {k: v for k, v in r.items() if k != "exit_reason_breakdown"}
            for r in sensitivity_results
        ])
        print("\n=== Z-Score Threshold Sensitivity ===")
        print(sens_df)

In [ ]:
# Test different max holding periods
if len(coint_pairs) >= 1:
    hold_periods = [60, 120, 180]
    hold_results = []

    for hold_min in hold_periods:
        print(f"\nTesting max_holding = {hold_min} min...")
        trades = backtest_all_pairs(
            coint_pairs, timeframe="1min",
            max_holding_minutes=hold_min,
        )
        if len(trades) > 0:
            eq = build_equity_curve(trades)
            m = calculate_all_metrics(trades, eq["equity"])
            m["max_holding_min"] = hold_min
            hold_results.append(m)

    if hold_results:
        hold_df = pl.DataFrame([
            {k: v for k, v in r.items() if k != "exit_reason_breakdown"}
            for r in hold_results
        ])
        print("\n=== Holding Period Sensitivity ===")
        print(hold_df)

## 9. Conclusions & Next Steps

### Summary
- Identified cointegrated pairs within sector groups using Engle-Granger ADF test
- Backtested intraday mean-reversion strategy on 1-minute data
- Evaluated performance across multiple parameter settings

### Ideas for Improvement
- **Dynamic hedge ratio**: Re-estimate hedge ratio using rolling window instead of static
- **Regime detection**: Add volatility regime filter to avoid trading during high-vol periods
- **Kalman filter**: Use Kalman filter for spread estimation instead of rolling OLS
- **Volume filters**: Only enter trades when volume is above average
- **Cross-sector pairs**: Test pairs across sectors for diversification
- **Slippage modeling**: Add realistic slippage based on bid-ask spread and volume
- **Walk-forward validation**: Split data into in-sample/out-of-sample periods
- **Position sizing**: Implement Kelly criterion or risk-parity sizing
- **Execution optimization**: Use VWAP or TWAP for order execution
- **Live trading**: Connect to broker API for paper/live trading